In [16]:
import pandas as pd
import torch
import numpy as np

In [43]:
data = pd.read_csv('./data/train.csv')
data

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8688,9276_01,Europa,False,A/98/P,55 Cancri e,41.0,True,0.0,6819.0,0.0,1643.0,74.0,Gravior Noxnuther,False
8689,9278_01,Earth,True,G/1499/S,PSO J318.5-22,18.0,False,0.0,0.0,0.0,0.0,0.0,Kurta Mondalley,False
8690,9279_01,Earth,False,G/1500/S,TRAPPIST-1e,26.0,False,0.0,0.0,1872.0,1.0,0.0,Fayey Connon,True
8691,9280_01,Europa,False,E/608/S,55 Cancri e,32.0,False,0.0,1049.0,0.0,353.0,3235.0,Celeon Hontichre,False


In [64]:
class Dataset(torch.utils.data.Dataset):
  def __init__(self, csv) -> None:
    super().__init__()
    df = pd.read_csv(csv)
    # I am skipping Cabin because I want to keep this simple
    self.y = df['Transported']
    df = df.drop(['PassengerId', 'Cabin', 'Name', 'Transported'], axis=1)
    self.home_planet = pd.get_dummies(df['HomePlanet'], prefix='HomePlanet')
    self.destination = pd.get_dummies(df['Destination'], prefix='Destination')
    df = df.drop(['HomePlanet', 'Destination'], axis=1)
    df['CryoSleep'] = df['CryoSleep'].fillna(False)
    df['VIP'] = df['VIP'].fillna(False)
    self.x = df.fillna(0).astype('uint8')


  def __len__(self):
    return len(self.y)
  
  def __getitem__(self, idx):
    if isinstance(idx, torch.Tensor):
      idx = idx.tolist()
      
    x = torch.from_numpy(np.concatenate([
        self.home_planet.iloc[idx].values,
        self.destination.iloc[idx].values,
        self.x.iloc[idx].values
    ]))
     
    return [x, self.y.iloc[idx]]



In [66]:
dataset = Dataset('./data/train.csv')
x = dataset.__getitem__(5)[0]
x.shape

torch.Size([14])

In [81]:
from torch import nn
class Model(nn.Module):
  def __init__(self):
    super().__init__()
    features = 3 + 3 + 1 + 1 + 1 + 5
    self.layers = nn.Sequential(
      nn.Linear(features, features),
      nn.ReLU(),
      nn.Linear(features, 1)
    )

  def forward(self, x):
    return self.layers(x).squeeze()

In [86]:
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt


def train(Model, epochs=100):
  model = Model()
  criterion = nn.MSELoss()
  optimizer = torch.optim.Adam(model.parameters(), weight_decay=0.0001)
  
  dataset = Dataset('./data/train.csv')
  train_size = int(0.8 * len(dataset))
  test_size = len(dataset) - train_size
  trainset, testset = random_split(dataset, [train_size, test_size])

  train_loader = DataLoader(trainset, batch_size=200, shuffle=True)
  test_loader = DataLoader(testset, batch_size=200, shuffle=True)
  
  loss_per_iter = []
  loss_per_batch = []
  for epoch in range(epochs):
    print("Epoch", epoch)
    running_loss = 0.0
    for i, (inputs, labels) in enumerate(train_loader):
      outputs = model(inputs.float())
      loss = criterion(outputs, labels.float())
      loss.backward()
      optimizer.step()

      # Save loss to plot
      running_loss += loss.item()
      loss_per_iter.append(loss.item())

      loss_per_batch.append(running_loss / (i + 1))
      running_loss = 0.0
      
  # Comparing training to test
  dataiter = iter(test_loader)
  inputs, labels = dataiter.next()
  # outputs = net(inputs.float())
  print("Root mean squared error")
  print("Training:", np.sqrt(loss_per_batch[-1]))

  # Plot training loss curve
  plt.plot(np.arange(len(loss_per_iter)), loss_per_iter, "-", alpha=0.5, label="Loss per epoch")
  plt.plot(np.arange(len(loss_per_iter), step=4) + 3, loss_per_batch, ".-", label="Loss per mini-batch")
  plt.xlabel("Number of epochs")
  plt.ylabel("Loss")
  plt.legend()
  plt.show()



In [87]:
train(Model)